In [1]:
import os
import shutil
import pandas as pd

# Project paths
PROJECT_PATH = "/content/drive/MyDrive/trendpulse/trendpulse"

PART4_PATH = os.path.join(
    PROJECT_PATH,
    "part4_sql_spreadsheet"
)

SOURCE_PATH = os.path.join(
    PROJECT_PATH,
    "part3_visualization_eda",
    "data"
)

# Create folders
os.makedirs(os.path.join(PART4_PATH, "data"), exist_ok=True)
os.makedirs(os.path.join(PART4_PATH, "notebooks"), exist_ok=True)
os.makedirs(os.path.join(PART4_PATH, "spreadsheets"), exist_ok=True)

# Copy original datasets
files = [
    "Copy of USvideos.csv",
    "Copy of GBvideos.csv",
    "Copy of INvideos.csv",
    "Copy of categories.csv"
]

for filename in files:
    source = os.path.join(SOURCE_PATH, filename)
    destination = os.path.join(PART4_PATH, "data", filename)

    shutil.copy2(source, destination)

    print("Copied:", filename)

print("\nPart 4 folder structure created successfully!")

Copied: Copy of USvideos.csv
Copied: Copy of GBvideos.csv
Copied: Copy of INvideos.csv
Copied: Copy of categories.csv

Part 4 folder structure created successfully!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Prepare the SQL dataset

In [3]:
# Load the cleaned dataset from Part 3
CLEANED_PATH = os.path.join(
    PROJECT_PATH,
    "part3_visualization_eda",
    "data",
    "part3_cleaned_dataset.csv"
)

df_sql = pd.read_csv(CLEANED_PATH)

print("Dataset shape:", df_sql.shape)

print("\nColumns:")
print(df_sql.columns.tolist())

print("\nFirst 5 records:")
display(df_sql.head())

Dataset shape: (46000, 21)

Columns:
['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description', 'country', 'category_name', 'days_to_trend', 'first_trending_date', 'trend_day']

First 5 records:


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,category_name,days_to_trend,first_trending_date,trend_day
0,g_N-w8LeAi8,2017-11-14,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14 07:04:28+00:00,Monsoon|Robots,48555,2189.0,180,...,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,People & Blogs,0,2017-11-14,1
1,Ugyql2NlPAd,2017-11-14,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12 22:35:48+00:00,Winter,198385,1159.0,492,...,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,News & Politics,2,2017-11-14,1
2,8fE19jkxJ9n,2017-11-14,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13 08:40:22+00:00,Cricket,231656,8742.0,896,...,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,Music,1,2017-11-14,1
3,WOY0nn7HCsO,2017-11-14,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27 22:21:47+00:00,Festival|Guitar,8083,176.0,8,...,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,Music,18,2017-11-14,1
4,2Lc_Cs5VVr_,2017-11-14,Surprising Election Recipe,UltimateVlog96,27,2017-11-09 13:20:20+00:00,Festival,373161,11305.0,167,...,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,Education,5,2017-11-14,1


##Create a SQLite database

In [4]:
import sqlite3
import os
import pandas as pd

# Create the database
DB_PATH = os.path.join(PART4_PATH, "trendpulse.db")

conn = sqlite3.connect(DB_PATH)

# Store the dataset as a SQL table
df_sql.to_sql("videos", conn, if_exists="replace", index=False)

print("Database created successfully!")

# Check the number of records
result = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM videos",
    conn
)

display(result)

Database created successfully!


,total_records
0,46000


##Inspect the SQL table

In [5]:
# Display the structure of the videos table
table_info = pd.read_sql_query(
    "PRAGMA table_info(videos)",
    conn
)

display(table_info)

,cid,name,type,notnull,dflt_value,pk
0,0,video_id,TEXT,0,None,0
1,1,trending_date,TEXT,0,None,0
2,2,title,TEXT,0,None,0
3,3,channel_title,TEXT,0,None,0
4,4,category_id,INTEGER,0,None,0
5,5,publish_time,TEXT,0,None,0
6,6,tags,TEXT,0,None,0
7,7,views,INTEGER,0,None,0
8,8,likes,REAL,0,None,0
9,9,dislikes,INTEGER,0,None,0


##most popular categories

In [6]:
query = """
SELECT
    category_name,
    COUNT(*) AS total_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views
FROM videos
GROUP BY category_name
ORDER BY total_views DESC;
"""

category_analysis = pd.read_sql_query(query, conn)

display(category_analysis)

,category_name,total_videos,total_views,average_views
0,Music,10310,32898699419,3190950.48
1,Entertainment,12270,11055584710,901025.65
2,People & Blogs,3804,2096894597,551234.12
3,Comedy,3258,1966123716,603475.66
4,Film & Animation,2297,1327606333,577974.02
5,Gaming,2057,1048380780,509664.94
6,Sports,1921,720333053,374978.16
7,News & Politics,2422,687350879,283794.75
8,Education,1645,596668258,362716.27
9,Science & Technology,1562,556619345,356350.41


##Country-wise performance

In [7]:
query = """
SELECT
    country,
    COUNT(*) AS total_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views,
    SUM(likes) AS total_likes
FROM videos
GROUP BY country
ORDER BY total_views DESC;
"""

country_analysis = pd.read_sql_query(query, conn)

display(country_analysis)

,country,total_videos,total_views,average_views,total_likes
0,GB,15000,23760619750,1584041.32,978742379.0
1,US,16000,16222470043,1013904.38,632806436.0
2,IN,15000,14281511461,952100.76,497032012.0


##top 10 most-viewed videos

In [9]:
query = """
SELECT
    title,
    channel_title,
    country,
    category_name,
    views,
    likes
FROM videos
ORDER BY views DESC
LIMIT 10;
"""

top_videos = pd.read_sql_query(query, conn)

display(top_videos)

,title,channel_title,country,category_name,views,likes
0,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,Music,360760164,27537822.0
1,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,Music,343297002,9581910.0
2,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,Music,286714357,21885701.0
3,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,Music,282812154,7893691.0
4,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,Music,255362869,11413793.0
5,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,Music,236275528,18035565.0
6,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,Music,230704137,6439282.0
7,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,Music,210747294,16086924.0
8,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,Music,200444300,8959133.0
9,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,Music,181785943,5073905.0


##Channel performance analysis

In [10]:
query = """
SELECT
    channel_title,
    COUNT(DISTINCT video_id) AS unique_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views,
    SUM(likes) AS total_likes
FROM videos
GROUP BY channel_title
HAVING COUNT(DISTINCT video_id) >= 2
ORDER BY total_views DESC
LIMIT 10;
"""

channel_analysis = pd.read_sql_query(query, conn)

display(channel_analysis)

,channel_title,unique_videos,total_views,average_views,total_likes
0,QuietChallenge75,466,5669763684,2313245.08,239006247.0
1,WildUnboxing39,56,3019916986,7664763.92,87814006.0
2,FinalSpeech81,34,2362212889,12632154.49,169234280.0
3,PerfectTutorial78,98,2286945332,4250827.75,93306231.0
4,PerfectTrailer65,66,2253048658,6607180.82,89412377.0
5,FinalTrailer17,161,2048349982,2265873.87,88122113.0
6,SurprisingHighlights41,123,1820925979,2810070.96,82822086.0
7,QuietInterview9,89,1673862447,2962588.40,66935177.0
8,ShockingMystery18,67,1371767573,3250634.06,61742150.0
9,OfficialReview41,457,1361332444,863241.88,43031052.0


##Engagement rate analysis

In [11]:
query = """
SELECT
    category_name,
    SUM(views) AS total_views,
    SUM(likes) AS total_likes,
    SUM(comment_count) AS total_comments,
    ROUND(
        SUM(likes) * 100.0 / NULLIF(SUM(views), 0),
        2
    ) AS like_rate_percentage
FROM videos
GROUP BY category_name
ORDER BY like_rate_percentage DESC;
"""

engagement_analysis = pd.read_sql_query(query, conn)

display(engagement_analysis)

,category_name,total_views,total_likes,total_comments,like_rate_percentage
0,Comedy,1966123716,9.347858e+07,7939455.0,4.75
1,Music,32898699419,1.391234e+09,67570165.0,4.23
2,Gaming,1048380780,4.267713e+07,5687111.0,4.07
3,Pets & Animals,319320474,1.270752e+07,955912.0,3.98
4,People & Blogs,2096894597,7.545390e+07,11679205.0,3.60
5,Education,596668258,2.075270e+07,1297143.0,3.48
6,Howto & Style,546924713,1.864575e+07,1662239.0,3.41
7,Entertainment,11055584710,3.549985e+08,46420434.0,3.21
8,Science & Technology,556619345,1.773510e+07,1942604.0,3.19
9,Autos & Vehicles,281650521,8.766490e+06,807650.0,3.11


##Analyse how quickly videos become trending

In [12]:
query = """
SELECT
    category_name,
    ROUND(AVG(days_to_trend), 2) AS average_days_to_trend,
    MIN(days_to_trend) AS fastest_trending_days,
    MAX(days_to_trend) AS slowest_trending_days,
    COUNT(DISTINCT video_id) AS unique_videos
FROM videos
GROUP BY category_name
ORDER BY average_days_to_trend ASC;
"""

trend_speed_analysis = pd.read_sql_query(query, conn)

display(trend_speed_analysis)

,category_name,average_days_to_trend,fastest_trending_days,slowest_trending_days,unique_videos
0,News & Politics,3.41,0,32,1381
1,Travel & Events,3.77,0,31,328
2,Howto & Style,3.98,0,32,743
3,Autos & Vehicles,4.13,0,42,415
4,Pets & Animals,4.13,0,34,395
5,Gaming,4.19,0,33,767
6,Sports,4.25,0,34,797
7,Science & Technology,4.35,0,39,660
8,Education,4.42,0,38,654
9,People & Blogs,4.49,0,34,1331


##Country and category comparison (SQL JOIN)

In [13]:
query = """
SELECT
    country,
    category_name,
    COUNT(DISTINCT video_id) AS unique_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views
FROM videos
GROUP BY country, category_name
ORDER BY country, total_views DESC;
"""

country_category_analysis = pd.read_sql_query(query, conn)

display(country_category_analysis.head(20))

,country,category_name,unique_videos,total_views,average_views
0,GB,Music,841,18606204798,3728698.36
1,GB,Entertainment,695,2310209959,817484.06
2,GB,Film & Animation,225,566565822,797980.03
3,GB,People & Blogs,407,566091188,438151.07
4,GB,Comedy,313,472824458,504615.22
5,GB,Gaming,207,298706099,495366.67
6,GB,Sports,249,192318676,284495.08
7,GB,Science & Technology,183,156138041,332916.93
8,GB,Howto & Style,221,127935190,218319.44
9,GB,News & Politics,252,124709165,233975.92


##second SQL table for JOIN

In [14]:
category_df = df_sql[["category_id", "category_name"]].drop_duplicates()

print("Category table shape:", category_df.shape)

display(category_df.head(20))

Category table shape: (14, 2)


,category_id,category_name
0,22,People & Blogs
1,25,News & Politics
2,10,Music
4,27,Education
5,24,Entertainment
14,26,Howto & Style
16,23,Comedy
17,20,Gaming
18,19,Travel & Events
26,17,Sports


In [15]:
print("Are category IDs unique?", category_df["category_id"].is_unique)

Are category IDs unique? True


##Save the categories table in SQLite

In [16]:
category_df.to_sql(
    "categories",
    conn,
    if_exists="replace",
    index=False
)

print("Categories table created successfully!")

Categories table created successfully!


In [17]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

display(tables)

,name
0,videos
1,categories


##first SQL JOIN

In [18]:
query = """
SELECT
    c.category_name,
    COUNT(DISTINCT v.video_id) AS unique_videos,
    SUM(v.views) AS total_views,
    ROUND(AVG(v.views), 2) AS average_views
FROM videos v
INNER JOIN categories c
    ON v.category_id = c.category_id
GROUP BY c.category_name
ORDER BY total_views DESC;
"""

join_analysis = pd.read_sql_query(query, conn)

display(join_analysis)

,category_name,unique_videos,total_views,average_views
0,Music,1883,32898699419,3190950.48
1,Entertainment,3387,11055584710,901025.65
2,People & Blogs,1331,2096894597,551234.12
3,Comedy,1129,1966123716,603475.66
4,Film & Animation,829,1327606333,577974.02
5,Gaming,767,1048380780,509664.94
6,Sports,797,720333053,374978.16
7,News & Politics,1381,687350879,283794.75
8,Education,654,596668258,362716.27
9,Science & Technology,660,556619345,356350.41


##monthly trending performance

In [19]:
query = """
SELECT
    substr(trending_date, 1, 7) AS trend_month,
    country,
    COUNT(DISTINCT video_id) AS unique_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views
FROM videos
GROUP BY trend_month, country
ORDER BY trend_month, country;
"""

monthly_analysis = pd.read_sql_query(query, conn)

display(monthly_analysis.head(20))

,trend_month,country,unique_videos,total_views,average_views
0,2017-11,GB,1090,3166103414,931206.89
1,2017-11,IN,1414,2450985860,720878.19
2,2017-11,US,1171,2468155132,725927.98
3,2017-12,GB,1791,10665889487,1720304.76
4,2017-12,IN,2488,6967041659,1123716.40
5,2017-12,US,2031,7016221900,1131648.69
6,2018-01,GB,1513,9928626849,1838634.60
7,2018-01,IN,2245,4863483942,900645.17
8,2018-01,US,2065,6557296156,1057628.41
9,2018-02,US,200,180796855,903984.28


##Save SQL analysis results as CSV files

In [20]:
import os

OUTPUT_PATH = os.path.join(PART4_PATH, "data")
os.makedirs(OUTPUT_PATH, exist_ok=True)

analysis_results = {
    "category_analysis.csv": join_analysis,
    "country_category_analysis.csv": country_category_analysis,
    "monthly_analysis.csv": monthly_analysis
}

for filename, data in analysis_results.items():
    file_path = os.path.join(OUTPUT_PATH, filename)
    data.to_csv(file_path, index=False)
    print("Saved:", filename)

print("\nSQL analysis results saved successfully!")

Saved: category_analysis.csv
Saved: country_category_analysis.csv
Saved: monthly_analysis.csv

SQL analysis results saved successfully!


##Save the remaining SQL analysis results

In [21]:
queries = {
    "country_analysis.csv": """
        SELECT country,
               COUNT(*) AS total_videos,
               SUM(views) AS total_views,
               ROUND(AVG(views), 2) AS average_views,
               SUM(likes) AS total_likes
        FROM videos
        GROUP BY country
        ORDER BY total_views DESC
    """,

    "top_videos.csv": """
        SELECT title, channel_title, country,
               category_name, views, likes
        FROM videos
        ORDER BY views DESC
        LIMIT 10
    """,

    "channel_analysis.csv": """
        SELECT channel_title,
               COUNT(DISTINCT video_id) AS unique_videos,
               SUM(views) AS total_views,
               ROUND(AVG(views), 2) AS average_views,
               SUM(likes) AS total_likes
        FROM videos
        GROUP BY channel_title
        HAVING COUNT(DISTINCT video_id) >= 2
        ORDER BY total_views DESC
        LIMIT 10
    """,

    "engagement_analysis.csv": """
        SELECT category_name,
               SUM(views) AS total_views,
               SUM(likes) AS total_likes,
               SUM(comment_count) AS total_comments,
               ROUND(SUM(likes) * 100.0 /
                     NULLIF(SUM(views), 0), 2) AS like_rate_percentage
        FROM videos
        GROUP BY category_name
        ORDER BY like_rate_percentage DESC
    """,

    "trend_speed_analysis.csv": """
        SELECT category_name,
               ROUND(AVG(days_to_trend), 2) AS average_days_to_trend,
               MIN(days_to_trend) AS fastest_trending_days,
               MAX(days_to_trend) AS slowest_trending_days,
               COUNT(DISTINCT video_id) AS unique_videos
        FROM videos
        GROUP BY category_name
        ORDER BY average_days_to_trend ASC
    """
}

for filename, sql in queries.items():
    result_df = pd.read_sql_query(sql, conn)
    result_df.to_csv(os.path.join(OUTPUT_PATH, filename), index=False)
    print("Saved:", filename)

print("\nAll remaining SQL results saved!")

Saved: country_analysis.csv
Saved: top_videos.csv
Saved: channel_analysis.csv
Saved: engagement_analysis.csv
Saved: trend_speed_analysis.csv

All remaining SQL results saved!


##Create the Excel workbook

In [22]:
import pandas as pd
import os

EXCEL_PATH = os.path.join(
    PART4_PATH,
    "spreadsheets",
    "trendpulse_analysis.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_PATH), exist_ok=True)

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:

    join_analysis.to_excel(
        writer, sheet_name="Category Analysis", index=False
    )

    country_category_analysis.to_excel(
        writer, sheet_name="Country Category", index=False
    )

    monthly_analysis.to_excel(
        writer, sheet_name="Monthly Analysis", index=False
    )

    pd.read_csv(os.path.join(OUTPUT_PATH, "country_analysis.csv")).to_excel(
        writer, sheet_name="Country Analysis", index=False
    )

    pd.read_csv(os.path.join(OUTPUT_PATH, "top_videos.csv")).to_excel(
        writer, sheet_name="Top Videos", index=False
    )

    pd.read_csv(os.path.join(OUTPUT_PATH, "channel_analysis.csv")).to_excel(
        writer, sheet_name="Channel Analysis", index=False
    )

    pd.read_csv(os.path.join(OUTPUT_PATH, "engagement_analysis.csv")).to_excel(
        writer, sheet_name="Engagement", index=False
    )

    pd.read_csv(os.path.join(OUTPUT_PATH, "trend_speed_analysis.csv")).to_excel(
        writer, sheet_name="Trend Speed", index=False
    )

print("Excel workbook created successfully!")
print("Saved at:", EXCEL_PATH)

Excel workbook created successfully!
Saved at: /content/drive/MyDrive/trendpulse/trendpulse/part4_sql_spreadsheet/spreadsheets/trendpulse_analysis.xlsx


In [24]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb = load_workbook(EXCEL_PATH)

for ws in wb.worksheets:

    # Format header row
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(
            start_color="1F4E78",
            end_color="1F4E78",
            fill_type="solid"
        )
        cell.alignment = Alignment(horizontal="center")

    # Freeze header
    ws.freeze_panes = "A2"

    # Enable filters
    ws.auto_filter.ref = ws.dimensions

    # Adjust column widths
    for column in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)

        for cell in column:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[column_letter].width = min(max_length + 3, 45)

wb.save(EXCEL_PATH)

print("Excel workbook formatted successfully!")


Excel workbook formatted successfully!


##Add charts to the Excel workbook

In [25]:
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference

wb = load_workbook(EXCEL_PATH)

# Chart 1: Category Performance
ws = wb["Category Analysis"]

chart1 = BarChart()
chart1.type = "bar"
chart1.style = 10
chart1.title = "Total Views by Category"
chart1.x_axis.title = "Total Views"
chart1.y_axis.title = "Category"
chart1.height = 10
chart1.width = 18

data = Reference(
    ws,
    min_col=3,
    min_row=1,
    max_row=ws.max_row
)

categories = Reference(
    ws,
    min_col=1,
    min_row=2,
    max_row=ws.max_row
)

chart1.add_data(data, titles_from_data=True)
chart1.set_categories(categories)

ws.add_chart(chart1, "G2")


# Chart 2: Monthly Performance
ws2 = wb["Monthly Analysis"]

chart2 = BarChart()
chart2.type = "col"
chart2.style = 11
chart2.title = "Monthly Total Views by Country"
chart2.x_axis.title = "Month"
chart2.y_axis.title = "Total Views"
chart2.height = 10
chart2.width = 18

data2 = Reference(
    ws2,
    min_col=4,
    min_row=1,
    max_row=ws2.max_row
)

categories2 = Reference(
    ws2,
    min_col=1,
    min_row=2,
    max_row=ws2.max_row
)

chart2.add_data(data2, titles_from_data=True)
chart2.set_categories(categories2)

ws2.add_chart(chart2, "G2")

wb.save(EXCEL_PATH)

print("Both Excel charts added successfully!")

Both Excel charts added successfully!


Verify

In [26]:
for root, dirs, files in os.walk(PART4_PATH):
    level = root.replace(PART4_PATH, "").count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

part4_sql_spreadsheet/
    README.md
    requirements.txt
    analysis.sql
    trendpulse.db
    data/
        Copy of INvideos.csv
        Copy of categories.csv
        Copy of cleaned_video_records.csv
        Copy of GBvideos.csv
        Copy of combined_videos.csv
        Copy of USvideos.csv
        category_analysis.csv
        country_category_analysis.csv
        monthly_analysis.csv
        country_analysis.csv
        top_videos.csv
        channel_analysis.csv
        engagement_analysis.csv
        trend_speed_analysis.csv
    notebooks/
    spreadsheets/
        trendpulse_analysis.xlsx


##Create the analysis.sql file

In [27]:
sql_content = """

-- ==========================================
-- TRENDPULSE
-- SQL AND SPREADSHEET ANALYSIS
-- ==========================================

-- 1. Total records
SELECT COUNT(*) AS total_records
FROM videos;


-- 2. Category performance
SELECT
    category_name,
    COUNT(*) AS total_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views
FROM videos
GROUP BY category_name
ORDER BY total_views DESC;


-- 3. Country performance
SELECT
    country,
    COUNT(*) AS total_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views,
    SUM(likes) AS total_likes
FROM videos
GROUP BY country
ORDER BY total_views DESC;


-- 4. Top 10 trending records by views
SELECT
    title,
    channel_title,
    country,
    category_name,
    views,
    likes
FROM videos
ORDER BY views DESC
LIMIT 10;


-- 5. Channel performance
SELECT
    channel_title,
    COUNT(DISTINCT video_id) AS unique_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views,
    SUM(likes) AS total_likes
FROM videos
GROUP BY channel_title
HAVING COUNT(DISTINCT video_id) >= 2
ORDER BY total_views DESC
LIMIT 10;


-- 6. Category engagement
SELECT
    category_name,
    SUM(views) AS total_views,
    SUM(likes) AS total_likes,
    SUM(comment_count) AS total_comments,
    ROUND(
        SUM(likes) * 100.0 / NULLIF(SUM(views), 0),
        2
    ) AS like_rate_percentage
FROM videos
GROUP BY category_name
ORDER BY like_rate_percentage DESC;


-- 7. Trending speed by category
SELECT
    category_name,
    ROUND(AVG(days_to_trend), 2) AS average_days_to_trend,
    MIN(days_to_trend) AS fastest_trending_days,
    MAX(days_to_trend) AS slowest_trending_days,
    COUNT(DISTINCT video_id) AS unique_videos
FROM videos
GROUP BY category_name
ORDER BY average_days_to_trend ASC;


-- 8. Country and category performance
SELECT
    country,
    category_name,
    COUNT(DISTINCT video_id) AS unique_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views
FROM videos
GROUP BY country, category_name
ORDER BY country, total_views DESC;


-- 9. INNER JOIN: Videos and categories
SELECT
    c.category_name,
    COUNT(DISTINCT v.video_id) AS unique_videos,
    SUM(v.views) AS total_views,
    ROUND(AVG(v.views), 2) AS average_views
FROM videos v
INNER JOIN categories c
    ON v.category_id = c.category_id
GROUP BY c.category_name
ORDER BY total_views DESC;


-- 10. Monthly trending performance
SELECT
    substr(trending_date, 1, 7) AS trend_month,
    country,
    COUNT(DISTINCT video_id) AS unique_videos,
    SUM(views) AS total_views,
    ROUND(AVG(views), 2) AS average_views
FROM videos
GROUP BY trend_month, country
ORDER BY trend_month, country;

"""

SQL_PATH = os.path.join(PART4_PATH, "analysis.sql")

with open(SQL_PATH, "w", encoding="utf-8") as file:
    file.write(sql_content)

print("analysis.sql created successfully!")

analysis.sql created successfully!


##Create requirements.txt

In [28]:
requirements = """pandas
numpy
matplotlib
seaborn
openpyxl
"""

REQUIREMENTS_PATH = os.path.join(PART4_PATH, "requirements.txt")

with open(REQUIREMENTS_PATH, "w") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


##README.md

In [31]:
readme_content = '''
# TrendPulse — What's Actually Trending Right Now?

## Project Overview
TrendPulse analyzes YouTube trending video data from the United States,
Great Britain, and India using Python, SQL, and Excel.

## Project Objectives
- Analyze YouTube trending videos across three countries.
- Identify popular video categories.
- Compare country performance.
- Analyze channel performance and engagement.
- Study how quickly videos become trending.
- Present insights using Excel charts.

## Tools and Technologies
- Python
- Pandas and NumPy
- Matplotlib and Seaborn
- SQLite and SQL
- Microsoft Excel
- OpenPyXL
- Google Colab

## Dataset
The cleaned dataset contains 46,000 records and 21 columns.

Countries: US, GB, and IN.

## SQL Analysis
1. Total records
2. Category performance
3. Country performance
4. Top 10 trending records
5. Channel performance
6. Category engagement
7. Trending speed
8. Country and category analysis
9. INNER JOIN
10. Monthly trending performance

## Key Findings
- Music recorded the highest accumulated views across categories.
- Entertainment had the most trending records.
- Great Britain had the highest accumulated views.
- Comedy had the highest like rate.
- News & Politics had the lowest average days to trend.
- Music had the highest average days to trend.

## Project Structure
- README.md
- requirements.txt
- analysis.sql
- trendpulse.db
- data/
- notebooks/
- spreadsheets/

## How to Run
1. Open the notebook in Google Colab or Jupyter.
2. Install dependencies using pip install -r requirements.txt.
3. Run the notebook cells in order.
4. Explore the SQL queries in analysis.sql.
5. Open the Excel workbook to view the analysis.

## Important Data Note
Some videos appear in multiple daily trending records.
Therefore, SUM(views) represents accumulated values across
the available records, not unique lifetime video views.

The top 10 results are trending records, so a video may
appear more than once.

## Conclusion
TrendPulse demonstrates an end-to-end data analytics workflow
using data preparation, SQL querying, exploratory analysis,
spreadsheet reporting, and data-driven insights.
'''

README_PATH = os.path.join(PART4_PATH, "README.md")

with open(README_PATH, "w", encoding="utf-8") as file:
    file.write(readme_content)

print("README.md created successfully!")

README.md created successfully!
